# Norway vs. US Labor Market — Exploratory Analysis

This notebook walks through the data before building the dashboard.
The goal is to understand what we're working with, spot any data quality issues,
and identify the key findings worth highlighting.

**Before running:** make sure `data/processed/` has the CSV files.
If not, run `python generate_sample_data.py` from the project root.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os

PROCESSED_DIR = "../data/processed"

# Load the three clean datasets
unemployment = pd.read_csv(f"{PROCESSED_DIR}/unemployment_clean.csv")
wages        = pd.read_csv(f"{PROCESSED_DIR}/wages_clean.csv")
employment   = pd.read_csv(f"{PROCESSED_DIR}/employment_clean.csv")

print("unemployment:", unemployment.shape)
print("wages:",        wages.shape)
print("employment:",   employment.shape)

## 1. Data Overview

Check shapes, column names, data types, and missing values before doing any analysis.

In [ ]:
print("=== Unemployment ===")
print(unemployment.dtypes)
print("\nMissing values:")
print(unemployment.isnull().sum())
print("\nYear range:", unemployment['year'].min(), "to", unemployment['year'].max())
unemployment.head(8)

In [ ]:
print("=== Wages ===")
print(wages.dtypes)
print("\nIndustries covered:", wages['industry'].unique())
print("\nMissing wage values:", wages['wage_annual_usd_ppp'].isnull().sum())
wages[wages['industry'] == 'Technology'].describe()

In [ ]:
print("=== Employment ===")
print(employment['industry'].value_counts())
employment[employment['industry'] == 'Technology']

## 2. Key Finding 1: Unemployment Resilience

Norway's unemployment rate stayed much lower than the US throughout 2010-2024,
including during the COVID-19 shock. 

Hypothesis: Norway's active labor market policies (keeping people in employment
through subsidies) and generous unemployment benefits cushioned the shock
compared to the US layoff-heavy model.

In [ ]:
fig = px.line(
    unemployment,
    x='year', y='unemployment_rate', color='country',
    color_discrete_map={'Norway': '#003087', 'United States': '#B22234'},
    markers=True,
    title='Annual Unemployment Rate: Norway vs. United States (2010-2024)',
    labels={'unemployment_rate': 'Unemployment Rate (%)', 'year': 'Year'}
)
fig.add_vrect(x0=2020, x1=2021, fillcolor='rgba(255,200,0,0.15)',
              line_width=0, annotation_text='COVID-19')
fig.update_layout(yaxis_ticksuffix='%', hovermode='x unified')
fig.show()

# Quantify the COVID gap
covid_year = unemployment[unemployment['year'] == 2020].set_index('country')['unemployment_rate']
print(f"\n2020 unemployment rates:")
print(covid_year)
print(f"US-Norway gap in 2020: {covid_year['United States'] - covid_year['Norway']:.1f} percentage points")

## 3. Key Finding 2: Tech Wages (PPP-Adjusted)

Are US tech workers paid more or less than Norwegian tech workers once we account
for cost-of-living differences?

We use OECD PPP conversion factors to normalize wages to internationally
comparable 'international dollars.'

In [ ]:
tech_wages = wages[wages['industry'] == 'Technology'].copy()

fig = px.line(
    tech_wages,
    x='year', y='wage_annual_usd_ppp', color='country',
    color_discrete_map={'Norway': '#003087', 'United States': '#B22234'},
    markers=True,
    title='Tech Sector Annual Wages, PPP-Adjusted USD (2010-2024)',
    labels={'wage_annual_usd_ppp': 'Annual Wage (PPP USD)', 'year': 'Year'}
)
fig.update_layout(yaxis_tickprefix='$', yaxis_tickformat=',')
fig.show()

# 2023 comparison
wages_2023 = tech_wages[tech_wages['year'] == 2023].set_index('country')['wage_annual_usd_ppp']
print("\n2023 tech wages (PPP-adjusted annual USD):")
print(wages_2023)
if 'United States' in wages_2023.index and 'Norway' in wages_2023.index:
    diff_pct = (wages_2023['United States'] / wages_2023['Norway'] - 1) * 100
    print(f"\nUS tech wages are {diff_pct:.1f}% {'higher' if diff_pct > 0 else 'lower'} than Norway (PPP-adjusted)")

## 4. Key Finding 3: The Tech Wage Premium

How much more do tech workers earn compared to the national average in each country?
A higher premium suggests tech is a more 'elite' sector in that economy.

In [ ]:
tech_w  = wages[wages['industry'] == 'Technology'][['country','year','wage_annual_usd_ppp']].rename(columns={'wage_annual_usd_ppp': 'tech_wage'})
total_w = wages[wages['industry'] == 'Total'][['country','year','wage_annual_usd_ppp']].rename(columns={'wage_annual_usd_ppp': 'avg_wage'})

merged = tech_w.merge(total_w, on=['country','year'])
merged['premium_ratio'] = (merged['tech_wage'] / merged['avg_wage']).round(2)

fig = px.line(
    merged,
    x='year', y='premium_ratio', color='country',
    color_discrete_map={'Norway': '#003087', 'United States': '#B22234'},
    markers=True,
    title='Tech Wage Premium (Tech Wage / National Average Wage)',
    labels={'premium_ratio': 'Premium Ratio', 'year': 'Year'}
)
fig.add_hline(y=1.0, line_dash='dash', line_color='gray', annotation_text='National average')
fig.show()

print("\n2023 tech wage premium:")
print(merged[merged['year']==2023][['country','tech_wage','avg_wage','premium_ratio']].to_string(index=False))

## 5. Tech Employment Share

What percentage of each country's workforce works in tech?

In [ ]:
tech_emp = employment[employment['industry'] == 'Technology'].copy()

fig = px.line(
    tech_emp,
    x='year', y='employment_pct', color='country',
    color_discrete_map={'Norway': '#003087', 'United States': '#B22234'},
    markers=True,
    title='Tech Sector as % of Total Workforce (2010-2024)',
    labels={'employment_pct': '% of Total Workforce', 'year': 'Year'}
)
fig.update_layout(yaxis_ticksuffix='%')
fig.show()

# Latest year
latest = tech_emp['year'].max()
print(f"\n{latest} tech employment share:")
print(tech_emp[tech_emp['year']==latest][['country','employment_pct','employment_count']].to_string(index=False))

## 6. Limitations

Every analysis has limitations. Documenting them shows analytical maturity.

1. **Industry classification mismatch:** Norway's NACE J (Information & Communication)
   includes publishing and broadcasting, while the US NAICS 51 (Information) has a
   slightly different scope. Comparisons are indicative, not exact.

2. **PPP assumptions:** We use OECD PPP factors for GDP. Tech workers' actual
   purchasing power might differ from economy-wide averages (they may live in
   more expensive cities). PPP is an approximation.

3. **Wage data coverage:** SSB wages cover full-time employees in registered
   enterprises. BLS wages cover production and nonsupervisory workers in some
   series. Gig workers and self-employed are not fully captured in either.

4. **Time granularity:** We aggregate monthly BLS data to annual averages.
   This smooths out seasonal variation but also hides the peak of COVID
   unemployment (April 2020: 14.7% in the US) in the annual average.

5. **Data availability:** Some SSB tables only go back to 2011 or 2015 for
   certain industry breakdowns. Where data is unavailable, we document it
   rather than interpolate.